## 6 — MRdeeP (state-level estimates, CES sample1)
Multivariate Multilevel Regression with Deep Generative Post-Stratification.
Outcomes extracted: `climate_problem`, `renewable_fuel`.

Pipeline:
1. `insert_data` — encodes CES survey + county-level benchmark
2. `fit` — trains an ensemble of CGANs (Wasserstein loss + gradient penalty)
3. `post_stratify('state_fips')` — generates synthetic micro-data per demographic
   cell, groups by state → extracts `climate_problem` and `renewable_fuel` estimates

In [2]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.environ['DEEPVERSE_BACKEND'] = 'pytorch'
sys.path.insert(0, '/Users/carmenk/Documents/CSS/Capstone/mrdeep/python')
from mrdeep import MRdeeP

sys.path.insert(0, str(Path('.').resolve()))
from utils import OUTPUT_DIR, STATE_FIPS_TO_NAME, SURVEY_PATH, save_estimates

DATA_DIR   = Path("../../")
OUTCOME    = ['climate_problem', 'renewable_fuel']
MODEL_NAME = 'mrdeep'

### 1. Load and prepare data

In [3]:
OUTCOME_COLS = ['climate_problem','regulate_carbon','renewable_fuel',
                'clean_air_water','fuel_efficiency','fossil_fuel','paris_agreement']
DEMOG_VARS   = ['gender', 'race4', 'educ_category', 'county_fips', 'state_fips']

raw       = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str, 'county_fips': str})
ps_county = pd.read_csv(DATA_DIR / 'post_stratification_frame' / 'poststrat_county.csv',
                        dtype={'state_fips': str, 'county_fips': str})

raw['county_fips'] = raw['county_fips'].astype(str).str.zfill(5)
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in raw.columns]

survey = raw[DEMOG_VARS + OUTCOME_COLS].dropna().copy()
survey['educ_category'] = survey['educ_category'].astype(str)

benchmark = ps_county[DEMOG_VARS + ['N_rounded']].copy()
benchmark['educ_category'] = benchmark['educ_category'].astype(str)
target_rows = len(benchmark)
benchmark['count'] = np.maximum(
    1,
    (benchmark['N_rounded'] / benchmark['N_rounded'].sum() * target_rows).round(),
).astype(int)
benchmark = benchmark.drop(columns=['N_rounded'])

print(f'Survey (complete cases): {len(survey):,}')
for oc in OUTCOME_COLS:
    print(f'  {oc}: {survey[oc].mean()*100:.1f}% support')
print(f'Benchmark strata: {len(benchmark):,}  augmented rows: {benchmark["count"].sum():,}')

Survey (complete cases): 988
  climate_problem: 65.1% support
  regulate_carbon: 67.0% support
  renewable_fuel: 63.2% support
  clean_air_water: 59.9% support
  fuel_efficiency: 68.2% support
  fossil_fuel: 62.1% support
  paris_agreement: 61.5% support
Benchmark strata: 99,940  augmented rows: 170,690


### 2. Insert data into MRdeeP

In [4]:
mod = MRdeeP(ensembles=3, random_state=42)

mod.insert_data(
    survey     = survey,
    benchmark  = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = 'count',
    oversample = 1,
)
print(mod)

MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 988 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: False


### 3. Train CGAN ensemble
Default architecture: 4 × 256-neuron hidden layers, Wasserstein loss + gradient penalty.

In [5]:
mod.fit(
    epochs        = 500,
    patience      = 50,
    batch_size    = 256,
    k             = 32,
    print_runtime = True,
)
print(mod)

Ensemble 1/3
Ensemble 2/3
Ensemble 3/3
Total fit time: 131.2 seconds.
MRdeeP (backend=pytorch, ensembles=3)
  Data inserted: True
  Survey: 988 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: True
    Noise dim (k): 32
    Ensemble members: 3
    Generated survey: 170690 rows
    Total fit time: 131.2s


### 4. Post-stratify → state-level estimates for all outcomes

In [6]:
estimates = mod.post_stratify(levels='state_fips')
print(f'Estimates shape: {estimates.shape}  ({estimates["state_fips"].nunique()} states)')
estimates.head()

Estimates shape: (51, 8)  (51 states)


,state_fips,clean_air_water,climate_problem,fossil_fuel,fuel_efficiency,paris_agreement,regulate_carbon,renewable_fuel
0,01,0.521087,0.581282,0.597576,0.637828,0.617924,0.680181,0.635195
1,02,0.523486,0.583251,0.597222,0.635339,0.616210,0.678968,0.634256
2,04,0.525634,0.589905,0.594819,0.639377,0.612458,0.676897,0.637106
3,05,0.521715,0.582432,0.598076,0.638664,0.615515,0.678477,0.635613
4,06,0.524390,0.581382,0.596447,0.636013,0.615858,0.678327,0.635633


### 5. Extract target outcomes and save

In [7]:
for OUTCOME_VAR in OUTCOME:
    result = estimates[['state_fips', OUTCOME_VAR]].rename(
        columns={OUTCOME_VAR: 'estimate'}
    ).copy()
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)

    save_estimates(result, MODEL_NAME, OUTCOME_VAR)

    print(f'\n--- {OUTCOME_VAR} ---')
    print(f'National mean: {result["estimate"].mean():.3f}')
    print(result.sort_values("estimate", ascending=False).head(5)[["state_name","estimate"]].to_string(index=False))


  mrdeep (climate_problem) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.5822
  Median estimate:       0.5824
  Min estimate:          0.5775
  Max estimate:          0.5899

  Saved → /Users/carmenk/Documents/GitHub/MRdeeP-Deep-Learning-MRP/model_run_ces/sample1_state/outputs/estimates/climate_problem_state_estimates.csv


--- climate_problem ---
National mean: 0.582
state_name  estimate
   Arizona  0.589905
     Maine  0.586094
  Delaware  0.585346
    Nevada  0.584687
Washington  0.584623

  mrdeep (renewable_fuel) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.6354
  Median estimate:       0.6354
  Min estimate:          0.6304
  Max estimate:          0.6441

  Saved → /Users/carmenk/Documents/GitHub/MRdeeP-Deep-Learning-MRP/model_run_ces/sample1_state/outputs/estimates/renewable_fuel_state_estimates.csv


--- renewable_fuel ---
National mean: 0.635
          sta